# GSM8K benchmark on Fireworks (Eval Protocol)

Bare-bones notebook: call a Fireworks model on GSM8K rows, grade with **regex** or an **LLM judge**, print accuracy.

**Prereqs:** Jupyter kernel = conda `cookbook` env, `FIREWORKS_API_KEY` in `training/.env`. Run the install cell below once if imports fail.

In [1]:
# Run once if imports fail (uses the notebook kernel's Python).
import sys
!{sys.executable} -m pip install -q -e "../../.[eval]"

In [2]:
# --- edit these ---
MODEL = "fireworks_ai/accounts/fireworks/models/minimax-m3"
JUDGE_MODEL = "fireworks_ai/accounts/fireworks/models/llama-v3p1-8b-instruct"  # only used when GRADING_MODE="llm_judge"

GRADING_MODE = "regex"  # "regex" or "llm_judge"
MAX_ROWS = None  # smoke test: 2-3 rows. Set to None for the full dataset.

DATASET_URL = "https://raw.githubusercontent.com/eval-protocol/python-sdk/main/development/gsm8k_sample.jsonl"
TEMPERATURE = 0.0

Example row:

```json
{
  "messages": [
    {
      "role": "system",
      "content": "\nYou are a highly capable and helpful math problem-solving assistant. Your goal is to carefully analyze and solve the user's mathematics question by providing a detailed, step-by-step explanation that is easy to follow and logically sound.\n\n## Instructions\n\n1. **Carefully read and fully understand the user's question.** Identify what is being asked and any relevant information or constraints.\n\n2. **Generate a detailed problem-solving process**, breaking down the solution into clear, logical steps. Explain each step thoroughly, including all intermediate calculations, reasoning, definitions, and justifications—even if they seem simple or obvious. This helps ensure clarity and aids understanding.\n\n3. **Structure your reasoning by enclosing all these problem-solving steps within `<think>` tags.**\n\n4. **After completing the detailed reasoning, provide the final answer clearly and unambiguously, enclosed within `<answer>` tags.** Ensure the answer directly addresses the question.\n\n5. If the problem involves multiple parts or sub-questions, address each part sequentially, separating your reasoning and answers clearly.\n\n6. Use correct mathematical notation and terminology throughout.\n\n7. Avoid skipping steps or assuming knowledge; assume the reader needs a comprehensive explanation.\n\n## Output Format\n<think>\n<!-- Detailed, step-by-step problem-solving explanation goes here -->\n</think>\n\n<answer>\n<!-- Final answer to the problem goes here -->\n</answer>\n"
    },
    {
      "role": "user",
      "content": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
    },
    {
      "role": "assistant",
      "content": "<think>\nNatalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n</think>\n\n<answer>\n72\n</answer>"
    }
  ],
  "ground_truth": "<think>\nNatalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n</think>\n\n<answer>\n72\n</answer>"
}```

In [3]:
import asyncio
import os
import re
from pathlib import Path

import litellm
from dotenv import load_dotenv
from eval_protocol.common_utils import load_jsonl
from eval_protocol.models import EvaluateResult, EvaluationRow, InputMetadata
from eval_protocol.pytest import SingleTurnRolloutProcessor
from eval_protocol.pytest.types import RolloutProcessorConfig

training_dir = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "training" and (p / "pyproject.toml").exists()),
    Path("../../").resolve(),
)
load_dotenv(training_dir / ".env")

if not os.getenv("FIREWORKS_API_KEY"):
    raise EnvironmentError(f"Set FIREWORKS_API_KEY in {training_dir / '.env'}")

litellm.drop_params = False  # strict mode: bad params will raise errors

/opt/homebrew/Caskroom/miniconda/base/envs/cookbook/lib/python3.12/site-packages/eval_protocol/models.py:1156: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class TaskDefinitionModel(BaseModel):


In [4]:
def load_rows(url: str, max_rows: int | None) -> list[EvaluationRow]:
    raw = load_jsonl(url)
    rows = [EvaluationRow(**item) for item in raw]
    if max_rows is not None:
        rows = rows[:max_rows]
    for i, row in enumerate(rows):
        row.input_metadata.row_id = row.input_metadata.row_id or f"row-{i}"
    return rows


rows = load_rows(DATASET_URL, MAX_ROWS)
print(f"Loaded {len(rows)} rows ({'smoke test' if MAX_ROWS else 'full run'})")

Loaded 1000 rows (full run)


In [5]:
_ANSWER_TAG_RE = re.compile(r"<answer>\s*(.*?)\s*</answer>", re.IGNORECASE | re.DOTALL)
_DIGITS_RE = re.compile(r"(-?\d+(?:\.\d+)?)")


def extract_answer(text: str) -> str | None:
    """Pull the numeric answer from <answer> tags (Eval Protocol GSM8K format)."""
    if not text:
        return None
    m = _ANSWER_TAG_RE.search(text)
    chunk = m.group(1) if m else text
    digits = _DIGITS_RE.findall(chunk.replace(",", ""))
    return digits[-1] if digits else None


def user_question(row: EvaluationRow) -> str:
    for msg in row.messages:
        if msg.role == "user":
            return str(msg.content)
    return ""


def model_response(row: EvaluationRow) -> str:
    return str(row.messages[-1].content)


def grade_regex(row: EvaluationRow) -> EvaluateResult:
    pred = extract_answer(model_response(row))
    gt = extract_answer(str(row.ground_truth))
    if pred is None or gt is None:
        return EvaluateResult(score=0.0, reason=f"missing answer (pred={pred}, gt={gt})")
    ok = pred == gt
    return EvaluateResult(
        score=1.0 if ok else 0.0,
        reason=f"pred={pred}, gt={gt}",
    )

async def grade_llm_judge(row: EvaluationRow) -> EvaluateResult:
    prompt = (
        "Grade this math answer. Reply with exactly one word: CORRECT or INCORRECT.\n\n"
        f"Question: {user_question(row)}\n"
        f"Model answer: {model_response(row)}\n"
        f"Reference answer: {row.ground_truth}"
    )
    resp = await litellm.acompletion(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperkkkature=0.0,
        
    )
    verdict = (resp.choices[0].message.content or "").strip().upper()
    ok = "CORRECT" in verdict and "INCORRECT" not in verdict
    return EvaluateResult(score=1.0 if ok else 0.0, reason=verdict)

In [6]:
async def run_benchmark(rows: list[EvaluationRow]) -> list[EvaluationRow]:
    processor = SingleTurnRolloutProcessor(
        drop_trailing_assistant_messages=True  # True (default) -> do not send final assistant message in messages
    )
    config = RolloutProcessorConfig(
        # for minimax reasoning options are 'low', 'medium', 'high', 'xhigh', 'max', 'none' or 'adaptive'
        completion_params={"model": MODEL, "temperature": TEMPERATURE, "reasoning_effort": "adaptive"},
        mcp_config_path="",  # MCP config is used to test agents
        semaphore=asyncio.Semaphore(4),  # limits to 4 concurrent requests at a time
    )

    rollout_tasks = processor(rows, config)
    completed = await asyncio.gather(*rollout_tasks)

    graded: list[EvaluationRow] = []
    for row in completed:
        if GRADING_MODE == "regex":
            row.evaluation_result = grade_regex(row)
        elif GRADING_MODE == "llm_judge":
            row.evaluation_result = await grade_llm_judge(row)
        else:
            raise ValueError(f"Unknown GRADING_MODE: {GRADING_MODE}")
        graded.append(row)
    return graded


results = await run_benchmark(rows)

In [7]:
scores = [r.evaluation_result.score for r in results if r.evaluation_result is not None]
accuracy = sum(scores) / len(scores) if scores else 0.0

print(f"Model: {MODEL}")
print(f"Grading: {GRADING_MODE}")
print(f"Rows: {len(results)}")
print(f"Accuracy: {accuracy:.1%} ({int(sum(scores))}/{len(scores)})")
print()

for row in results[:5]:
    q = user_question(row)[:80].replace("\n", " ")
    ans = extract_answer(model_response(row))
    gt = extract_answer(str(row.ground_truth))
    score = row.evaluation_result.score if row.evaluation_result else float("nan")
    print(f"[{score:.0f}] pred={ans} gt={gt} | {q}...")

Model: fireworks_ai/accounts/fireworks/models/minimax-m3
Grading: regex
Rows: 1000
Accuracy: 80.0% (800/1000)

[1] pred=72 gt=72 | Natalia sold clips to 48 of her friends in April, and then she sold half as many...
[0] pred=50 gt=10 | Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of ba...
[1] pred=5 gt=5 | Betty is saving money for a new wallet which costs $100. Betty has only half of ...
[1] pred=42 gt=42 | Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and t...
[1] pred=624 gt=624 | James writes a 3-page letter to 2 different friends twice a week.  How many page...


## Full run

After the smoke test looks good, go back to the config cell, set `MAX_ROWS = None`, and re-run all cells below it.